# Feature Engineering and Target Construction

This notebook isolates target creation and modeling-table construction. The current cells preserve the existing baseline behavior and add the first ticker-hour aggregation needed for the next modeling pass.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_merged_data, summarize_data_quality
from src.plots import set_plot_style

set_plot_style()

from src.feature_engineering import add_next_close_target, aggregate_to_ticker_hour
from src.config import TABLES_DIR

TABLES_DIR.mkdir(parents=True, exist_ok=True)

df = load_merged_data()
display(df.head())


## Current Baseline Target

This is the target used in the original notebook: `1` when the next close for the same ticker is greater than the current close. It is kept here so the baseline remains reproducible.

In [ ]:
baseline_df = add_next_close_target(df)

print("Post-level target balance:")
print(baseline_df["Target_Direction"].value_counts().sort("Target_Direction"))


## Ticker-Hour Aggregation

The raw dataset has one row per post. For final modeling, one row per ticker-hour is cleaner because every post in the same ticker-hour shares the same market price target.

In [ ]:
hourly_df = aggregate_to_ticker_hour(df)

print(f"Raw rows: {df.height:,}")
print(f"Ticker-hour rows: {hourly_df.height:,}")
display(hourly_df.head())


In [ ]:
hourly_target_df = add_next_close_target(hourly_df)

print("Hourly target balance:")
print(hourly_target_df["Target_Direction"].value_counts().sort("Target_Direction"))


## Save Intermediate Modeling Table

This table is an intermediate baseline artifact. The next modeling iteration should replace or extend this with the final hybrid intraday/overnight target and richer features.

In [ ]:
output_path = TABLES_DIR / "hourly_baseline_modeling_dataset.csv"
hourly_target_df.write_csv(output_path)
print(f"Saved {hourly_target_df.height:,} rows to {output_path}")


## Next Feature Engineering Work

- Filter out timestamps outside valid Yahoo Finance hourly coverage.
- Add intraday/overnight target logic.
- Add lag returns, rolling returns, volume anomaly, sentiment EMA, sentiment z-score, hour/day features, and ticker encoding.
- Apply any scaling or sampling only after the train/test split in the modeling notebook.